In [2]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Panama boundary
panama = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Panama'))
)

# (Alternative) Panama boundary
# countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")

# panama = countries.filter(
#     ee.Filter.eq('country_na', 'Panama')
# ).geometry()

# Date range
START = '2024-01-01'
END = '2024-12-31'

        # Dynamic World dataset

dw_col = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterBounds(panama)
    .filterDate(START, END)
)

# Sentinel-2 collection
s2_col = (
    ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
    .filterBounds(panama)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

# print("DW images:", dw_col.size().getInfo())
# print("S2 images:", s2_col.size().getInfo())

# Create mosaics/composites
# Dynamic World label mode composite
dw_mode = dw_col.select('label').mode()

# Sentinel-2 median composite
s2_median = s2_col.median()

# Visualization palettes
VIS_PALETTE = [
    '419bdf',  # water
    '397d49',  # trees
    '88b053',  # grass
    '7a87c6',  # flooded vegetation
    'e49635',  # crops
    'dfc35a',  # shrub/scrub
    'c4281b',  # built
    'a59b8f',  # bare
    'b39fe1',  # snow/ice
]

# Visualize Dynamic World
dw_vis = dw_mode.visualize(
    min=0,
    max=8,
    palette=VIS_PALETTE
)

# Sentinel-2 RGB
Map.addLayer(
    s2_median,
    {
        'bands': ['B4', 'B3', 'B2'],
        'min': 0,
        'max': 3000
    },
    'Sentinel-2 Median'
)

# Dynamic World
Map.addLayer(
    dw_vis,
    {},
    'Dynamic World Land Cover'
)

# Panama boundary
Map.addLayer(
    panama,
    {'color': 'red'},
    'Panama Boundary'
)

Map.addLayerControl()

        # MERIT Topography

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

roi = panama.geometry()

# Sentinel-2 cloud mask
def mask_s2_clouds(image):

    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)

# Sentinel-2 collection
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2024-01-01", "2025-05-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_clouds)
)

# Composite clipped to Panama
image = s2.median().clip(roi)

# Visualization parameters
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3,
}

# Load DEM and clip to Panama
dem = ee.Image("MERIT/DEM/v1_0_3").clip(roi)

# Elevation visualization
dem_vis = {
    "min": 0,
    "max": 3500,
    "palette": [
        "#000000",
        "#478fcd",
        "#86c58e",
        "#afc35e",
        "#8f7131",
        "#b78d4f",
        "#e2b8a6",
        "#ffffff"   # high mountains
    ],
}

# Add layers
Map.addLayer(image, rgb_vis, "Sentinel-2")

Map.addLayer(dem, dem_vis, "Elevation")

# Center map
Map.centerObject(roi, 7)

        # MERIT DEM Slope Data

# Units in meters, 30m resolution
slope = ee.Image("projects/deforestation-495419/assets/panama_slopee")

# Clip to Panama
slope_clipped = slope.clip(panama)

Map.addLayer(slope_clipped, {}, 'slope clipped')

        # Soil layers

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

# Convert Panama feature collection to geometry
panama_geom = panama.geometry()

# Soil layers clipped to Panama
# Organic carbon (dg/kg)
pa = (
    ee.Image("projects/deforestation-495419/assets/Soil_Org_C")
    .clip(panama_geom)
)

# Nitrogen (cg/kg)
pn = (
    ee.Image("projects/deforestation-495419/assets/Soil_N")
    .clip(panama_geom)
)

# Sand (g/kg)
ps = (
    ee.Image("projects/deforestation-495419/assets/sand")
    .clip(panama_geom)
)

# pH (*10)
ph = (
    ee.Image("projects/deforestation-495419/assets/pH")
    .clip(panama_geom)
)

# Visualization parameters
pa_vis = {
    "min": 0,
    "max": 150,
    "palette": [
        "#fbd1cc",
        "#dd9889",
        "#e2685d",
        "#bc3535",
        "#680000"
    ]
}

pn_vis = {
    "min": 0,
    "max": 80,
    "palette": [
        "#f7fcf5",
        "#c7e9c0",
        "#74c476",
        "#238b45",
        "#00441b"
    ]
}

ps_vis = {
    "min": 0,
    "max": 1000,
    "palette": [
        "#fff5eb",
        "#fdd0a2",
        "#fdae6b",
        "#e6550d",
        "#7f2704"
    ]
}

ph_vis = {
    "min": 40,
    "max": 80,
    "palette": [
        "#d73027",
        "#fc8d59",
        "#fee08b",
        "#d9ef8b",
        "#1a9850"
    ]
}

# Add layers
Map.addLayer(pa, pa_vis, "Soil Organic Carbon")
Map.addLayer(pn, pn_vis, "Nitrogen")
Map.addLayer(ps, ps_vis, "Sand")
Map.addLayer(ph, ph_vis, "pH")

        # Distance to roads

roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

# Rasterize roads
roads_raster = ee.Image().byte().paint(roads, 1)

# Distance to nearest road in meters
distance_to_roads = (
    roads_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)  # adjust if your dataset resolution differs
    .rename("dist_roads_m")
    .clip(panama)
)

vis = {
    "min": 0,
    "max": 5000,
    "palette": ["white", "blue", "green", "yellow", "red"]
}

Map.addLayer(
    roads_raster,
    {"palette": ["black"]},
    "Roads (raster)"
)

Map.addLayer(
    distance_to_roads,
    vis,
    "Distance to Roads (m)"
)

        # Dry vs wet forests

# ESA WorldCover
worldcover = ee.ImageCollection(
    "ESA/WorldCover/v200"
).first()

landcover = worldcover.select('Map')

# Forest class in ESA WorldCover
# 10 = Tree cover
forest = landcover.eq(10)

# WWF Ecoregions
ecoregions = ee.FeatureCollection(
    "RESOLVE/ECOREGIONS/2017"
)

# Panama dry forests
dry_ecoregion = ecoregions.filter(
    ee.Filter.eq(
        'ECO_NAME',
        'Panamanian dry forests'
    )
)

# Rasterize dry forest polygon
dry_mask = ee.Image.constant(0).paint(
    dry_ecoregion,
    1
).selfMask()

# Dry forest
dry_forest = forest.updateMask(
    dry_mask
)

# Wet forest
wet_mask = dry_mask.unmask(0).eq(0)

wet_forest = forest.updateMask(
    wet_mask
)

# Create classification
classified = ee.Image(0)

# Wet forest = 1
classified = classified.where(
    wet_forest,
    1
)

# Dry forest = 2
classified = classified.where(
    dry_forest,
    2
)

Map.addLayer(
    classified.clip(panama),
    {
        'min': 0,
        'max': 2,
        'palette': [
            'white',    # non-forest
            '006400',   # wet forest
            'd4a017'    # dry forest
        ]
    },
    'Forest Type'
)

        # Distance to a deforested area

# Hansen dataset (clipped)
dataset = ee.Image('UMD/hansen/global_forest_change_2025_v1_13').clip(panama)

lossyear = dataset.select('lossyear')

# Define "recent loss"
# Example: last 5 years in dataset timeline
# Hansen lossyear is typically:
# 1 = 2001, ..., 25 = 2025
recent_threshold = 20  # ~2020 onward

recent_loss = lossyear.gte(recent_threshold).selfMask()

# Distance to recent loss
distance_recent = (
    recent_loss
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename('dist_recent_loss_m')
    .clip(panama)
)

# Visualization
vis = {
    'min': 0,
    'max': 100,  # 0.1 km range?
    'palette': ['red', 'orange', 'yellow', 'green', 'blue']
}

Map.add_layer(
    recent_loss,
    {'palette': ['ff0000']},
    'Recent forest loss (binary)'
)

Map.add_layer(
    distance_recent,
    vis,
    'Distance to recent forest loss (m)'
)

        # Distance to urban and rural areas

# Load GHSL SMOD dataset
image = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030")
smod = image.select("smod_code")

# Panama boundary
panama = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "Panama"))
    .geometry()
)

smod = smod.clip(panama)

# Urban / Rural masks
urban = smod.gte(21).And(smod.lte(30))
rural = smod.gte(11).And(smod.lte(13))

urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

# Distance surfaces
distance_to_urban = urban_img.distance(
    ee.Kernel.euclidean(50000, "meters")
).clip(panama)

distance_to_rural = rural_img.distance(
    ee.Kernel.euclidean(50000, "meters")
).clip(panama)

# Visualization
urban_distance_vis = {
    "min": 0,
    "max": 25000,
    "palette": [
        "#fff5eb",
        "#fd8d3c",
        "#d94801"
    ]
}

rural_distance_vis = {
    "min": 0,
    "max": 25000,
    "palette": [
        "#f7fbff",
        "#6baed6",
        "#08306b"
    ]
}

urban_mask_vis = {
    "palette": ["#e31a1c"]
}

rural_mask_vis = {
    "palette": ["#1f78b4"]
}

# Basemap
# Map.add_basemap("CartoDB.Positron")

# Distance layers
Map.addLayer(
    distance_to_urban,
    urban_distance_vis,
    "Distance to Urban",
    True,
    0.8
)

Map.addLayer(
    distance_to_rural,
    rural_distance_vis,
    "Distance to Rural",
    True,
    0.8
)

# Reference masks
Map.addLayer(
    urban.selfMask(),
    urban_mask_vis,
    "Urban Areas",
    True
)

Map.addLayer(
    rural.selfMask(),
    rural_mask_vis,
    "Rural Areas",
    False
)

# Layer control
Map.addLayerControl()

        # Precipitation data

# Load CHIRPS daily precipitation dataset
dataset = (
    ee.ImageCollection('UCSB-CHC/CHIRPS/V3/DAILY_SAT')
    .filter(ee.Filter.date('2018-05-01', '2018-05-31'))
)

# Select precipitation band and clip each image to Panama
precipitation = dataset.select('precipitation').map(
    lambda img: img.clip(panama)
)

# Visualization parameters
precipitation_vis = {
    'min': 1.0,
    'max': 17.0,
    'palette': [
        '#001137',
        '#0aab1e',
        '#e7eb05',
        '#2c7fb8',
        '#253494'
    ],
}

# Add clipped precipitation layer
# mm/day
Map.add_layer(
    precipitation,
    precipitation_vis,
    'Panama Precipitation'
)

        # Temperature data

# Load ERA5-Land monthly aggregated dataset and filter by date
dataset = (
    ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
    .filterDate('2023-01-01', '2023-02-01')
    .first()
)

# Clip to Panama
temperature = dataset.clip(panama)

# Visualization parameters
visualization = {
    'bands': ['temperature_2m'],
    'min': 250,
    'max': 320,
    'palette': [
        '000080', '0000d9', '4000ff', '8000ff',
        '0080ff', '00ffff', '00ff80', '80ff00',
        'daff00', 'ffff00', 'fff500', 'ffda00',
        'ffb000', 'ffa400', 'ff4f00', 'ff2500',
        'ff0a00', 'ff00ff',
    ]
}

# Add clipped temperature layer
Map.add_layer(
    temperature,
    visualization,
    'Air temperature [K] at 2m height',
    True,
    0.8
)

        # Distance to nearest hydrographic basin

# Hydrographic basins
basins = ee.Image(
    "projects/deforestation-495419/assets/hydrographic_basins"
).clip(panama)

# Convert to integer basin IDs
basins_int = basins.toInt()

# Detect basin boundaries

# Compare neighboring pixels
neighbors = basins_int.focal_mode(radius=1)

# Boundary pixels = where neighbor ID differs
basin_edges = basins_int.neq(neighbors).selfMask()

# Distance to nearest basin boundary

# Compute distance from NON-edge pixels to nearest edge
distance_pixels = basin_edges.fastDistanceTransform(
    neighborhood=512,
    units='pixels'
).sqrt()

# Convert to meters
pixel_size = basins.projection().nominalScale()

distance_to_basin_m = (
    distance_pixels
    .multiply(pixel_size)
    .rename("distance_to_basin_m")
    .clip(panama)
)

# Visualization

Map.addLayer(
    basins,
    {},
    "Hydrographic Basins"
)

Map.addLayer(
    basin_edges,
    {"palette": ["red"]},
    "Basin Boundaries"
)

Map.addLayer(
    distance_to_basin_m,
    {
        "min": 0,
        "max": 50000,
        "palette": ["blue", "cyan", "yellow", "red"]
    },
    "Distance to Basin Boundary (m)"
)

        # Distance from the nearest protected area (m)

pa = ee.FeatureCollection("WCMC/WDPA/current/polygons")

# Rasterize protected area polygons
pa_raster = ee.Image().byte().paint(pa, 1)

# distance to nearest protected area in meters
distance_to_pa = (
    pa_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)  # adjust if your dataset resolution differs
    .rename("dist_pa_m")
    .clip(panama)
)

Map.addLayer(pa, {}, 'Protected Areas')
Map.addLayer(distance_to_pa, {}, 'Distance to Protected Areas (m)')

# Map
Map = geemap.Map()

Map.centerObject(panama, 7)

Map

Map(center=[8.513881071215703, -80.10553238925856], controls=(WidgetControl(options=['position', 'transparent_…